# Chinese → Vietnamese Final Kaggle Notebook

This notebook is a **single complete file** for the UITAIC Chinese→Vietnamese challenge.

It does all of this:
- loads the dataset from Kaggle
- trains a non-pretrained Transformer
- uses exact-match translation memory
- decodes test sentences in **fast batched greedy mode**
- exports the official `submission.zip` with columns:
  - `tieng_trung`
  - `tieng_viet`

Recommended Kaggle accelerator:
- **GPU T4**
- avoid **P100** in this setup

In [ ]:
from __future__ import annotations

import os
import math
import json
import time
import random
import zipfile
from pathlib import Path
from dataclasses import dataclass, asdict
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

try:
    import sacrebleu
except Exception:
    sacrebleu = None

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True

def pick_device():
    if torch.cuda.is_available():
        try:
            major, minor = torch.cuda.get_device_capability(0)
            name = torch.cuda.get_device_name(0)
            print(f"Detected GPU: {name} (sm_{major}{minor})")
            if major >= 7:
                print("Using CUDA")
                return torch.device("cuda")
            print("GPU not compatible with this PyTorch build, using CPU.")
        except Exception as e:
            print("CUDA inspection failed:", e)
    else:
        print("CUDA not available.")
    return torch.device("cpu")

DEVICE = pick_device()
USE_AMP = DEVICE.type == "cuda"
IS_KAGGLE = Path("/kaggle").exists()

@dataclass
class CFG:
    data_root_override: str = "/kaggle/input/datasets/accelra/chinese-language-translate-to-vietnamese"
    valid_ratio: float = 0.08
    max_src_len: int = 64
    max_tgt_len: int = 64
    min_freq_src: int = 1
    min_freq_tgt: int = 1

    d_model: int = 256
    nhead: int = 8
    num_encoder_layers: int = 4
    num_decoder_layers: int = 4
    dim_feedforward: int = 768
    dropout: float = 0.15

    batch_size: int = 160 if DEVICE.type == "cuda" else 48
    infer_batch_size: int = 192 if DEVICE.type == "cuda" else 48
    epochs: int = 8 if DEVICE.type == "cuda" else 3
    lr: float = 3e-4
    weight_decay: float = 1e-4
    label_smoothing: float = 0.1
    grad_clip: float = 1.0
    num_workers: int = 2 if DEVICE.type == "cuda" else 0
    early_stopping: int = 3

    max_decode_len: int = 64
    save_dir: str = "artifacts_mt_final_clean"
    run_name: str = "zh_vi_transformer_exact_memory_official"

cfg = CFG()
print("Device:", DEVICE)
print("Kaggle:", IS_KAGGLE)
print("SacreBLEU available:", sacrebleu is not None)
print(json.dumps(asdict(cfg), indent=2))

In [ ]:
def has_required_files(root: Path):
    return (
        (root / "dataset/train/train.zh").exists()
        and (root / "dataset/train/train.vi").exists()
        and (root / "dataset/test/test.zh").exists()
    )

def find_data_root():
    candidates = []
    override = Path(cfg.data_root_override)
    if override.exists():
        candidates.append(override)

    candidates.extend([
        Path("."),
        Path("/kaggle/input"),
        Path("/kaggle/working"),
        Path("/mnt/data"),
    ])

    for root in candidates:
        if root.exists() and has_required_files(root):
            return root

    for base in [Path("/kaggle/input"), Path("/kaggle/working"), Path("."), Path("/mnt/data")]:
        if not base.exists():
            continue
        for p in base.rglob("train.zh"):
            candidate = p.parent.parent.parent
            if has_required_files(candidate):
                return candidate
    raise FileNotFoundError("Could not find dataset root with dataset/train/train.zh etc.")

DATA_ROOT = find_data_root()
TRAIN_SRC = DATA_ROOT / "dataset/train/train.zh"
TRAIN_TGT = DATA_ROOT / "dataset/train/train.vi"
TEST_SRC = DATA_ROOT / "dataset/test/test.zh"

SAVE_DIR = Path("/kaggle/working" if IS_KAGGLE else ".") / cfg.save_dir
SAVE_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_ROOT =", DATA_ROOT)
print("TRAIN_SRC =", TRAIN_SRC)
print("TRAIN_TGT =", TRAIN_TGT)
print("TEST_SRC  =", TEST_SRC)
print("SAVE_DIR  =", SAVE_DIR)

In [ ]:
def read_lines(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return [line.rstrip("\n") for line in f if line.rstrip("\n") != ""]

train_src = read_lines(TRAIN_SRC)
train_tgt = read_lines(TRAIN_TGT)
test_src = read_lines(TEST_SRC)

assert len(train_src) == len(train_tgt)

print("Train pairs:", len(train_src))
print("Test size  :", len(test_src))
print()
print("Sample pair:")
print("ZH:", train_src[0])
print("VI:", train_tgt[0])

In [ ]:
def grouped_train_valid_split(src, tgt, valid_ratio=0.08, seed=42):
    src_to_idx = defaultdict(list)
    for i, s in enumerate(src):
        src_to_idx[s].append(i)

    uniq = list(src_to_idx.keys())
    rng = np.random.default_rng(seed)
    rng.shuffle(uniq)
    n_valid = max(1, int(len(uniq) * valid_ratio))
    valid_src_set = set(uniq[:n_valid])

    tr_src, tr_tgt, va_src, va_tgt = [], [], [], []
    for s, t in zip(src, tgt):
        if s in valid_src_set:
            va_src.append(s)
            va_tgt.append(t)
        else:
            tr_src.append(s)
            tr_tgt.append(t)
    return tr_src, tr_tgt, va_src, va_tgt

tr_src, tr_tgt, va_src, va_tgt = grouped_train_valid_split(train_src, train_tgt, cfg.valid_ratio, SEED)

print("Train:", len(tr_src))
print("Valid:", len(va_src))
print("Test :", len(test_src))

In [ ]:
class Vocab:
    def __init__(self, token_lists, min_freq=1):
        specials = ["<pad>", "<bos>", "<eos>", "<unk>"]
        counter = Counter()
        for toks in token_lists:
            counter.update(toks)
        self.itos = specials[:]
        for tok, freq in counter.items():
            if freq >= min_freq and tok not in specials:
                self.itos.append(tok)
        self.stoi = {tok: i for i, tok in enumerate(self.itos)}
        self.pad_id = self.stoi["<pad>"]
        self.bos_id = self.stoi["<bos>"]
        self.eos_id = self.stoi["<eos>"]
        self.unk_id = self.stoi["<unk>"]

    def __len__(self):
        return len(self.itos)

    def encode(self, tokens, add_bos=False, add_eos=False, max_len=None):
        if max_len is not None:
            tokens = tokens[:max_len]
        ids = [self.stoi.get(t, self.unk_id) for t in tokens]
        if add_bos:
            ids = [self.bos_id] + ids
        if add_eos:
            ids = ids + [self.eos_id]
        return ids

    def decode(self, ids, skip_specials=True):
        out = []
        specials = {"<pad>", "<bos>", "<eos>", "<unk>"}
        for idx in ids:
            tok = self.itos[idx]
            if skip_specials and tok in specials:
                continue
            out.append(tok)
        return out

src_token_lists = [s.split() for s in tr_src]
tgt_token_lists = [t.split() for t in tr_tgt]

src_vocab = Vocab(src_token_lists, min_freq=cfg.min_freq_src)
tgt_vocab = Vocab(tgt_token_lists, min_freq=cfg.min_freq_tgt)

PAD_ID = tgt_vocab.pad_id
BOS_ID = tgt_vocab.bos_id
EOS_ID = tgt_vocab.eos_id
UNK_ID = tgt_vocab.unk_id

print("Source vocab:", len(src_vocab))
print("Target vocab:", len(tgt_vocab))
print("Special token ids:", PAD_ID, BOS_ID, EOS_ID, UNK_ID)

In [ ]:
memory_table = defaultdict(list)
for s, t in zip(train_src, train_tgt):
    memory_table[s].append(t)

exact_match_map = {s: Counter(ts).most_common(1)[0][0] for s, ts in memory_table.items()}
num_dups = sum(1 for s, ts in memory_table.items() if len(ts) > 1)
num_conflicts = sum(1 for s, ts in memory_table.items() if len(set(ts)) > 1)

print("Unique source sentences:", len(memory_table))
print("Sources with duplicates :", num_dups)
print("Conflicting duplicates   :", num_conflicts)

In [ ]:
class TranslationDataset(Dataset):
    def __init__(self, src_texts, tgt_texts, src_vocab, tgt_vocab, cfg):
        self.src_texts = src_texts
        self.tgt_texts = tgt_texts
        self.src_vocab = src_vocab
        self.tgt_vocab = tgt_vocab
        self.cfg = cfg

    def __len__(self):
        return len(self.src_texts)

    def __getitem__(self, idx):
        src_tokens = self.src_texts[idx].split()
        tgt_tokens = self.tgt_texts[idx].split()
        src_ids = self.src_vocab.encode(src_tokens, add_bos=True, add_eos=True, max_len=self.cfg.max_src_len)
        tgt_ids = self.tgt_vocab.encode(tgt_tokens, add_bos=True, add_eos=True, max_len=self.cfg.max_tgt_len)
        return {
            "src_ids": torch.tensor(src_ids, dtype=torch.long),
            "tgt_ids": torch.tensor(tgt_ids, dtype=torch.long),
        }

def pad_1d(seqs, pad_id):
    max_len = max(len(x) for x in seqs)
    out = torch.full((len(seqs), max_len), pad_id, dtype=torch.long)
    for i, x in enumerate(seqs):
        out[i, :len(x)] = x
    return out

def collate_fn(batch):
    src_ids = pad_1d([x["src_ids"] for x in batch], src_vocab.pad_id)
    tgt_ids = pad_1d([x["tgt_ids"] for x in batch], tgt_vocab.pad_id)
    return {"src_ids": src_ids, "tgt_ids": tgt_ids}

train_ds = TranslationDataset(tr_src, tr_tgt, src_vocab, tgt_vocab, cfg)
valid_ds = TranslationDataset(va_src, va_tgt, src_vocab, tgt_vocab, cfg)

train_loader = DataLoader(
    train_ds,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=cfg.num_workers,
    collate_fn=collate_fn,
    pin_memory=(DEVICE.type == "cuda"),
)
valid_loader = DataLoader(
    valid_ds,
    batch_size=cfg.infer_batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    collate_fn=collate_fn,
    pin_memory=(DEVICE.type == "cuda"),
)

print("Train batches:", len(train_loader))
print("Valid batches:", len(valid_loader))

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=512):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

class Seq2SeqTransformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model, nhead, num_encoder_layers,
                 num_decoder_layers, dim_feedforward, dropout, pad_id):
        super().__init__()
        self.pad_id = pad_id
        self.d_model = d_model
        self.src_embed = nn.Embedding(src_vocab_size, d_model, padding_idx=pad_id)
        self.tgt_embed = nn.Embedding(tgt_vocab_size, d_model, padding_idx=pad_id)
        self.pos_encoder = PositionalEncoding(d_model, dropout=dropout)
        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
        )
        self.generator = nn.Linear(d_model, tgt_vocab_size)

    def make_tgt_mask(self, tgt_len, device):
        return torch.triu(torch.ones(tgt_len, tgt_len, device=device, dtype=torch.bool), diagonal=1)

    def encode(self, src_ids):
        src_key_padding_mask = src_ids.eq(self.pad_id)
        src_emb = self.pos_encoder(self.src_embed(src_ids) * math.sqrt(self.d_model))
        memory = self.transformer.encoder(src_emb, src_key_padding_mask=src_key_padding_mask)
        return memory, src_key_padding_mask

    def decode(self, tgt_ids, memory, src_key_padding_mask):
        tgt_key_padding_mask = tgt_ids.eq(self.pad_id)
        tgt_mask = self.make_tgt_mask(tgt_ids.size(1), tgt_ids.device)
        tgt_emb = self.pos_encoder(self.tgt_embed(tgt_ids) * math.sqrt(self.d_model))
        out = self.transformer.decoder(
            tgt=tgt_emb,
            memory=memory,
            tgt_mask=tgt_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
            memory_key_padding_mask=src_key_padding_mask,
        )
        return self.generator(out)

    def forward(self, src_ids, tgt_input_ids):
        memory, src_key_padding_mask = self.encode(src_ids)
        return self.decode(tgt_input_ids, memory, src_key_padding_mask)

model = Seq2SeqTransformer(
    src_vocab_size=len(src_vocab),
    tgt_vocab_size=len(tgt_vocab),
    d_model=cfg.d_model,
    nhead=cfg.nhead,
    num_encoder_layers=cfg.num_encoder_layers,
    num_decoder_layers=cfg.num_decoder_layers,
    dim_feedforward=cfg.dim_feedforward,
    dropout=cfg.dropout,
    pad_id=src_vocab.pad_id,
).to(DEVICE)

num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print("Trainable parameters:", f"{num_params:,}")

In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID, label_smoothing=cfg.label_smoothing)
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay, betas=(0.9, 0.98))
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=cfg.lr,
    epochs=cfg.epochs,
    steps_per_epoch=max(1, len(train_loader)),
    pct_start=0.1,
    div_factor=10,
    final_div_factor=50,
)
autocast_device = "cuda" if DEVICE.type == "cuda" else "cpu"
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

def run_train_epoch(model, loader):
    model.train()
    losses = []
    pbar = tqdm(loader, desc="train", leave=False)
    for batch in pbar:
        src_ids = batch["src_ids"].to(DEVICE)
        tgt_ids = batch["tgt_ids"].to(DEVICE)

        tgt_in = tgt_ids[:, :-1]
        tgt_out = tgt_ids[:, 1:]

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(autocast_device, enabled=USE_AMP):
            logits = model(src_ids, tgt_in)
            loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1))

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        losses.append(loss.item())
        pbar.set_postfix(loss=f"{np.mean(losses):.4f}")
    return float(np.mean(losses))

def fallback_corpus_bleu(hypotheses, references, max_n=4, smooth=1.0):
    def extract_ngrams(tokens, n):
        return [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]

    hyp_tokens = [h.split() for h in hypotheses]
    ref_tokens = [r.split() for r in references]

    p_ns = []
    for n in range(1, max_n + 1):
        clipped = 0
        total = 0
        for hyp, ref in zip(hyp_tokens, ref_tokens):
            hyp_ngrams = Counter(extract_ngrams(hyp, n))
            ref_ngrams = Counter(extract_ngrams(ref, n))
            clipped += sum(min(count, ref_ngrams[ng]) for ng, count in hyp_ngrams.items())
            total += sum(hyp_ngrams.values())
        p_ns.append((clipped + smooth) / (total + smooth))

    hyp_len = sum(len(x) for x in hyp_tokens)
    ref_len = sum(len(x) for x in ref_tokens)
    if hyp_len == 0:
        return 0.0
    bp = 1.0 if hyp_len > ref_len else math.exp(1 - ref_len / max(hyp_len, 1))
    return 100.0 * bp * math.exp(sum((1 / max_n) * math.log(max(p, 1e-12)) for p in p_ns))

In [ ]:
@torch.no_grad()
def greedy_decode_batch(model, src_batch_texts, batch_size=192):
    model.eval()
    out_preds = []

    for start in tqdm(range(0, len(src_batch_texts), batch_size), desc="decode", leave=False):
        texts = src_batch_texts[start:start + batch_size]

        exact_preds = [exact_match_map.get(x) for x in texts]
        pending_idx = [i for i, p in enumerate(exact_preds) if p is None]

        batch_out = exact_preds[:]
        if pending_idx:
            src_token_ids = []
            for i in pending_idx:
                toks = texts[i].split()[:cfg.max_src_len]
                ids = src_vocab.encode(toks, add_bos=True, add_eos=True)
                src_token_ids.append(torch.tensor(ids, dtype=torch.long))

            src_tensor = pad_1d(src_token_ids, src_vocab.pad_id).to(DEVICE)
            memory, src_key_padding_mask = model.encode(src_tensor)

            ys = torch.full((src_tensor.size(0), 1), BOS_ID, dtype=torch.long, device=DEVICE)
            finished = torch.zeros(src_tensor.size(0), dtype=torch.bool, device=DEVICE)

            for _ in range(cfg.max_decode_len):
                logits = model.decode(ys, memory, src_key_padding_mask)
                next_token = logits[:, -1, :].argmax(dim=-1)
                next_token = next_token.masked_fill(finished, EOS_ID)
                ys = torch.cat([ys, next_token.unsqueeze(1)], dim=1)
                finished = finished | next_token.eq(EOS_ID)
                if finished.all():
                    break

            pred_token_ids = ys[:, 1:].tolist()
            pred_texts = []
            for row in pred_token_ids:
                cut = []
                for tok in row:
                    if tok == EOS_ID:
                        break
                    if tok in (PAD_ID, BOS_ID):
                        continue
                    cut.append(tok)
                pred_texts.append(" ".join(tgt_vocab.decode(cut, skip_specials=True)).strip())

            for local_i, global_i in enumerate(pending_idx):
                batch_out[global_i] = pred_texts[local_i]

        out_preds.extend(batch_out)
    return out_preds

@torch.no_grad()
def evaluate_bleu(model, valid_src, valid_tgt):
    preds = greedy_decode_batch(model, valid_src, batch_size=cfg.infer_batch_size)
    if sacrebleu is not None:
        bleu = sacrebleu.corpus_bleu(preds, [valid_tgt], force=True, tokenize="none").score
    else:
        bleu = fallback_corpus_bleu(preds, valid_tgt)
    return bleu, preds

In [ ]:
best_bleu = -1.0
best_path = SAVE_DIR / f"{cfg.run_name}_best.pt"
history = []
patience = 0

for epoch in range(1, cfg.epochs + 1):
    t0 = time.time()
    train_loss = run_train_epoch(model, train_loader)
    val_bleu, _ = evaluate_bleu(model, va_src, va_tgt)
    minutes = (time.time() - t0) / 60.0

    row = {"epoch": epoch, "train_loss": train_loss, "val_bleu": val_bleu, "minutes": minutes}
    history.append(row)
    print(f"Epoch {epoch:02d} | loss={train_loss:.4f} | val_bleu={val_bleu:.2f} | time={minutes:.1f} min")

    if val_bleu > best_bleu:
        best_bleu = val_bleu
        patience = 0
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "src_vocab_itos": src_vocab.itos,
                "tgt_vocab_itos": tgt_vocab.itos,
                "cfg": asdict(cfg),
                "best_bleu": best_bleu,
                "epoch": epoch,
            },
            best_path,
        )
        print("  -> saved best checkpoint to", best_path)
    else:
        patience += 1
        if patience >= cfg.early_stopping:
            print("Early stopping triggered.")
            break

history_df = pd.DataFrame(history)
history_df

In [ ]:
ckpt = torch.load(best_path, map_location=DEVICE)
model.load_state_dict(ckpt["model_state_dict"])
print("Loaded best checkpoint from epoch", ckpt["epoch"])
print("Best validation BLEU:", round(float(ckpt["best_bleu"]), 4))

sample_src = va_src[:5]
sample_pred = greedy_decode_batch(model, sample_src, batch_size=min(cfg.infer_batch_size, len(sample_src)))
for s, p, g in zip(sample_src, sample_pred, va_tgt[:5]):
    print("ZH :", s)
    print("PR :", p)
    print("GT :", g)
    print("-" * 80)

In [ ]:
test_preds = greedy_decode_batch(model, test_src, batch_size=cfg.infer_batch_size)
submission_df = pd.DataFrame({
    "tieng_trung": test_src,
    "tieng_viet": test_preds,
})
print(submission_df.head(10))
print("rows =", len(submission_df))

In [ ]:
submission_csv = SAVE_DIR / "submission.csv"
submission_zip = SAVE_DIR / "submission.zip"

submission_df.to_csv(submission_csv, index=False, encoding="utf-8")
with zipfile.ZipFile(submission_zip, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(submission_csv, arcname="submission.csv")

print("Saved CSV:", submission_csv)
print("Saved ZIP:", submission_zip)
print("ZIP size :", f"{submission_zip.stat().st_size / 1024:.2f} KB")
print("Header   :", ",".join(submission_df.columns.tolist()))

## Upload this file

Upload:
- `submission.zip`

Inside it there is exactly:
- `submission.csv`

with the official columns:
- `tieng_trung`
- `tieng_viet`